# mammo-cad — Full Pipeline Visualization Notebook
### U-Net Segmentation + EfficientNet-B3 Classification + Grad-CAM

Ce notebook exécute le pipeline complet sur une ou plusieurs images et visualise **chaque étape** :
1. Image brute vs prétraitée
2. Carte de probabilité U-Net + masque binaire
3. Clustering des composantes connexes
4. Classification par cluster (TTA-16) + incertitude
5. Grad-CAM par région
6. Résultat final annoté
7. Rapport JSON

---
> **Prérequis** : `segment.py`, `classify.py`, `pipeline.py` dans `src/inference/`

In [ ]:
import sys
import json
import time
import numpy as np
import cv2
import torch
import matplotlib
matplotlib.rcParams['figure.dpi'] = 130
matplotlib.rcParams['axes.facecolor'] = '#0d1117'
matplotlib.rcParams['figure.facecolor'] = '#0d1117'
matplotlib.rcParams['text.color'] = 'white'
matplotlib.rcParams['axes.labelcolor'] = 'white'
matplotlib.rcParams['xtick.color'] = 'white'
matplotlib.rcParams['ytick.color'] = 'white'
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'Root    : {PROJECT_ROOT}')

In [ ]:
# ── CONFIGURATION — Adapter les chemins ici ───────────────────────────
SEG_CKPT   = PROJECT_ROOT / 'checkpoints' / 'unet_best.pth'
CLS_CKPT   = PROJECT_ROOT / 'checkpoints' / 'best_0.7717' / 'efficientnet_stage2_swa.pth'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'pipeline_viz'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Images à analyser — mettre autant d'images que voulu
# Prendre des images du dossier data/processed/inbreast/AllPng/
INPUT_IMAGES = sorted(
    (PROJECT_ROOT / 'data' / 'processed' / 'inbreast' / 'AllPng').glob('*.png')
)[:10]   # <-- changer ce nombre pour analyser plus d'images

# Paramètres pipeline
SEG_THR      = 0.45
SEG_TTA      = False      # True = plus lent, légèrement meilleur
CLS_TTA      = 16
CLUSTER_DIST = 60         # [FIX] was 100 — 60px ≈ 4mm, cohérent avec pipeline.py
DEVICE       = 'auto'

print(f'Seg checkpoint : {SEG_CKPT}')
print(f'Cls checkpoint : {CLS_CKPT}')
print(f'Output dir     : {OUTPUT_DIR}')
print(f'Images found   : {len(INPUT_IMAGES)}')
for p in INPUT_IMAGES:
    print(f'  {p.name}')

## 1 · Chargement des modèles

In [ ]:
from src.inference.segment import load_model as load_seg_model, run_segmentation_array
from src.inference.classify import MammoClassifier
from src.inference.pipeline import (
    GradCAM, _make_gradcam_overlay,
    _crop_bbox, _annotate_image, _build_legend, RegionResult
)

device_str = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Loading U-Net...')
seg_model, seg_device = load_seg_model(SEG_CKPT, device=device_str)

print('\nLoading EfficientNet-B3...')
clf = MammoClassifier(CLS_CKPT, device=device_str, tta=CLS_TTA)
print(f'\n  Threshold : {clf.threshold:.2f}')
print(f'  Val AUC   : {clf.val_auc:.4f}')

## 1.1 · Architectures des modèles

**U-Net (segmentation)** : 5 niveaux encodeur + 4 décodeur, base=64 — ~7.7M params
**EfficientNet-B3 (classification)** : backbone ImageNet + tête Linear(1536→256→1) — ~10.7M params


In [ ]:
# Résumé des modèles — nombre de paramètres et taille d'entrée
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, train

seg_total, seg_train = count_params(seg_model)
cls_total, cls_train = count_params(clf.model)

summary = [
    ('U-Net',           '256×256×1', seg_total, seg_train, '0.9642', 'topk_CE + SGD 0.99', '10-fold CV'),
    ('EfficientNet-B3', '224×224×3', cls_total, cls_train, '0.7933', 'BCE + SWA',          'Two-stage FT'),
]

print(f"{'Modèle':<18} {'Input':<12} {'Params (M)':<12} {'Trainable':<12} {'AUC test':<10} {'Loss':<22} {'Training':<15}")
print('─' * 105)
for name, inp, tot, tr, auc, loss, trn in summary:
    print(f"{name:<18} {inp:<12} {tot/1e6:<12.2f} {tr/1e6:<12.2f} {auc:<10} {loss:<22} {trn:<15}")

# Visualisation en camembert + barres
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), facecolor='#0d1117')

ax = axes[0]
ax.set_facecolor('#161b22')
names = ['U-Net', 'EfficientNet-B3']
params = [seg_total / 1e6, cls_total / 1e6]
colors = ['#4DA1FF', '#FF6B35']
bars = ax.barh(names, params, color=colors, height=0.5)
for bar, p in zip(bars, params):
    ax.text(p + 0.1, bar.get_y() + bar.get_height()/2,
            f'{p:.2f}M', va='center', color='white', fontsize=10)
ax.set_xlabel('Paramètres (millions)', color='white')
ax.set_title('Taille des modèles', color='#aaa', fontsize=11)
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_color('#444')

ax = axes[1]
ax.set_facecolor('#161b22')
metrics = {
    'U-Net AUC (seg CV)': 0.9642, 
    'EffNet-B3 AUC (cls test)': 0.7933,
    'Pipeline AUC (INbreast)': 0.6200
}
colors2 = ['#4DA1FF', '#FF6B35', '#FFD700']
bars = ax.bar(list(metrics.keys()), list(metrics.values()), color=colors2, width=0.55)
for bar, v in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.015,
            f'{v:.4f}', ha='center', color='white', fontsize=10)
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='#666', ls='--', lw=0.8, label='aléatoire')
ax.set_ylabel('AUC-ROC', color='white')
ax.set_title('Performance des modèles', color='#aaa', fontsize=11)
ax.tick_params(colors='white')
ax.legend(fontsize=8)
for sp in ax.spines.values(): sp.set_color('#444')

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'model_architectures.png'), dpi=130,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()


## 1.2 · Métriques & Évaluation

Affichage des courbes ROC, matrices de confusion et analyse par BI-RADS
(générées par les scripts d'évaluation dans `outputs/`).


In [ ]:
# Chargement des visualisations d'évaluation existantes
from PIL import Image as PILImage

eval_png = {
    'Classifier ROC + CM (CBIS-DDSM test)' : PROJECT_ROOT / 'outputs' / 'evaluation' / 'cls_roc_cm_tta.png',
    'Classifier Grad-CAM samples (TTA-16)' : PROJECT_ROOT / 'outputs' / 'evaluation' / 'cls_gradcam_tta.png',
    'INbreast ROC + distribution'           : PROJECT_ROOT / 'outputs' / 'inbreast_evaluation' / 'inbreast_roc_distribution.png',
    'INbreast confusion matrix'             : PROJECT_ROOT / 'outputs' / 'inbreast_evaluation' / 'inbreast_confusion_metrics.png',
    'INbreast per-BI-RADS'                  : PROJECT_ROOT / 'outputs' / 'inbreast_evaluation' / 'inbreast_birads_analysis.png',
}

available = [(t, p) for t, p in eval_png.items() if p.exists()]
print(f'Visualisations d\'évaluation trouvées : {len(available)}/{len(eval_png)}\n')

if available:
    n = len(available)
    fig, axes = plt.subplots(n, 1, figsize=(14, 5.5 * n), facecolor='#0d1117')
    if n == 1:
        axes = [axes]
    for ax, (title, path) in zip(axes, available):
        img = np.array(PILImage.open(path))
        ax.imshow(img)
        ax.set_title(title, color='#aaa', fontsize=11, pad=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Aucune visualisation trouvée — lancer scripts/evaluate_classifier.py d\'abord.')

# Tableau comparatif before/after framework integration
print('\n── Impact de l\'intégration medical-image-std (framework Hamza) ──\n')
rows = [
    ('Métrique',   'Avant framework', 'Après framework', 'Gain'),
    ('AUC',        '0.4117',          '0.6200',          '+0.208'),
    ('Accuracy',   '60.0%',           '70.0%',           '+10%'),
    ('Sensitivity','50.0% (10/20)',   '60.0% (12/20)',   '+10%'),
    ('Specificity','66.7% (20/30)',   '76.7% (23/30)',   '+10%'),
    ('F1 Score',   '0.500',           '0.6154',          '+0.115'),
]
for row in rows:
    print(f'  {row[0]:<14} {row[1]:<22} {row[2]:<22} {row[3]}')


## 1.3 · Pipeline de prétraitement — étape par étape

`preprocess_mammogram()` applique 4 transformations :
1. **Flip** vers gauche (heuristique intensité)
2. **Breast mask** (`medical_image.BreastMaskAlgorithm` — Otsu + largest CC)
3. **Crop** du fond noir
4. **CLAHE** (clip=2.0, tile=8×8) pour rehausser les microcalcifications


In [ ]:
# Démo du preprocessing — appel à chaque étape individuellement
from medical_image.algorithms.breast_mask import BreastMaskAlgorithm as _BrMask
from medical_image.data.in_memory_image import InMemoryImage as _InMemImg

sample_path = INPUT_IMAGES[6]
img0 = cv2.imread(str(sample_path), cv2.IMREAD_GRAYSCALE)

# Étape 1 — flip
H0, W0 = img0.shape
img_flipped = cv2.flip(img0, 1) if img0[:, W0//2:].sum() > img0[:, :W0//2].sum() else img0.copy()

# Étape 2 — breast mask (framework)
_tensor_in = _InMemImg(array=torch.from_numpy(img_flipped.astype(np.float32)))
_tensor_out = _tensor_in.clone()
_BrMask(mask_only=True, device='cpu')(_tensor_in, _tensor_out)
mask = (_tensor_out.pixel_data.numpy() > 0).astype(np.uint8) * 255
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
mask_dilated = cv2.dilate(mask, kernel, iterations=2)
img_masked = cv2.bitwise_and(img_flipped, img_flipped, mask=mask_dilated)

# Étape 3 — crop
cols = np.any(img_masked > 5, axis=0)
rows = np.any(img_masked > 5, axis=1)
if cols.any() and rows.any():
    c0, c1 = np.where(cols)[0][[0, -1]]
    r0, r1 = np.where(rows)[0][[0, -1]]
    img_cropped = img_masked[r0:r1+1, c0:c1+1]
else:
    img_cropped = img_masked

# Étape 4 — CLAHE
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_clahe = clahe.apply(img_cropped)

steps = [
    ('① Brut',                 img0),
    ('② Flip (L-facing)',      img_flipped),
    ('③ Breast mask',          mask_dilated),
    ('④ Masqué',               img_masked),
    ('⑤ Cropped',              img_cropped),
    ('⑥ CLAHE final',          img_clahe),
]

fig, axes = plt.subplots(1, 6, figsize=(22, 5), facecolor='#0d1117')
for ax, (title, im) in zip(axes, steps):
    ax.imshow(im, cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'{title}\n{im.shape[1]}×{im.shape[0]}', color='#aaa', fontsize=10)
    ax.axis('off')
fig.suptitle(f'Preprocessing pipeline — {sample_path.name}',
             color='white', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'preprocessing_steps.png'), dpi=130,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

# Histogrammes avant/après CLAHE
fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor='#0d1117')
for ax, img, title in [(axes[0], img_cropped, 'Avant CLAHE'),
                        (axes[1], img_clahe, 'Après CLAHE')]:
    ax.set_facecolor('#161b22')
    ax.hist(img[img > 5].flatten(), bins=60, color='#FF6B35', alpha=0.85)
    ax.set_title(title, color='#aaa', fontsize=10)
    ax.set_xlabel('Intensité', color='white')
    ax.set_ylabel('Pixels', color='white')
    ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_color('#444')
plt.tight_layout()
plt.show()


## 1.4 · Intégration framework `medical-image-std` (Hamza Gbada / LATIS)

Deux algorithmes du framework sont intégrés dans le pipeline :

1. **`BreastMaskAlgorithm`** — remplace la segmentation Otsu custom (section précédente)
2. **`FebdsAlgorithm(method='dog')`** — fallback quand U-Net détecte 0 régions

Le fallback FEBDS a récupéré **2 cas BIRADS-5** précédemment manqués sur INbreast
(AUC 0.4117 → 0.6200, gain +0.208).


In [ ]:
# Démo FEBDS — enhancement map (DoG) vs original
from medical_image.algorithms.FEBDS import FebdsAlgorithm as _FebdsAlg

sample_path = INPUT_IMAGES[7]
img_input = cv2.imread(str(sample_path), cv2.IMREAD_GRAYSCALE)

# FEBDS-DoG
_img_f = _InMemImg(array=torch.from_numpy(img_input.astype(np.float32) / 255.0))
_out_f = _img_f.clone()
_FebdsAlg(method='dog', device='cpu')(_img_f, _out_f)
febds_raw = _out_f.pixel_data.numpy()
febds_mask = (febds_raw * 255).astype(np.uint8)

# Filtrage interior (comme dans pipeline.py)
H_p, W_p = img_input.shape
breast_mask = (img_input > 5).astype(np.uint8) * 255
breast_interior = cv2.erode(breast_mask,
                             cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (41, 41)),
                             iterations=2)
febds_mask_filtered = cv2.bitwise_and(febds_mask, febds_mask, mask=breast_interior)

# Composantes connexes
n_f, lbl_f, st_f, _ = cv2.connectedComponentsWithStats(febds_mask_filtered, connectivity=8)
febds_cands = []
margin = 20
for i in range(1, n_f):
    a = int(st_f[i, cv2.CC_STAT_AREA])
    if not (30 < a < 8000): continue
    x = int(st_f[i, cv2.CC_STAT_LEFT]); y = int(st_f[i, cv2.CC_STAT_TOP])
    w = int(st_f[i, cv2.CC_STAT_WIDTH]); h = int(st_f[i, cv2.CC_STAT_HEIGHT])
    if x <= margin or y <= margin or (x+w) >= W_p-margin or (y+h) >= H_p-margin:
        continue
    febds_cands.append((a, x, y, x+w, y+h))
febds_cands.sort(key=lambda c: -c[0])
top_bboxes = febds_cands[:8]

print(f'FEBDS candidats bruts : {n_f - 1}')
print(f'Après filtrage (interior + edge + area): {len(febds_cands)}')
print(f'Top-8 retenus pour le fallback :\n')
for i, (a, x1, y1, x2, y2) in enumerate(top_bboxes):
    print(f'  R{i:02d}  area={a:5d}  bbox=({x1},{y1})-({x2},{y2})')

# Visualisation
fig, axes = plt.subplots(1, 4, figsize=(20, 6), facecolor='#0d1117')

axes[0].imshow(img_input, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('① Input', color='#aaa', fontsize=10)
axes[0].axis('off')

axes[1].imshow(febds_raw, cmap='hot', vmin=0, vmax=1)
axes[1].set_title('② FEBDS-DoG enhancement', color='#aaa', fontsize=10)
axes[1].axis('off')

axes[2].imshow(febds_mask_filtered, cmap='gray')
axes[2].set_title('③ Masque filtré (interior)', color='#aaa', fontsize=10)
axes[2].axis('off')

overlay = cv2.cvtColor(img_input, cv2.COLOR_GRAY2BGR)
for _, x1, y1, x2, y2 in top_bboxes:
    cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 136), 3)
axes[3].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
axes[3].set_title(f'④ Top-{len(top_bboxes)} candidats FEBDS\n(si U-Net=0)',
                   color='#aaa', fontsize=10)
axes[3].axis('off')

fig.suptitle(f'Framework medical-image-std — Démo FEBDS fallback — {sample_path.name}',
             color='white', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'framework_febds_demo.png'), dpi=130,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

print('\nIntégration dans pipeline.py :')
print('  - preprocess_mammogram() utilise BreastMaskAlgorithm (Otsu + largest CC)')
print('  - run_pipeline() appelle FEBDS-DoG quand len(seg_result.bboxes) == 0')
print('  - Filtrage : area ∈ [30, 8000], margin=20px, masque intérieur (érosion 41×41 × 2)')


## 2 · Exécution du pipeline sur chaque image
On stocke tous les résultats intermédiaires pour la visualisation.

In [ ]:
_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

all_results = []   # list of dicts, one per image

for img_path in INPUT_IMAGES:
    print(f'\n{"─"*55}')
    print(f'Processing: {img_path.name}')
    t0 = time.perf_counter()

    # ── Load raw ──────────────────────────────────────────────────────
    img_raw = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)

    # ── [FIX] Images de data/processed/ sont déjà prétraitées ────────
    # NE PAS appeler preprocess_mammogram() — double preprocessing = faux positifs
    img_proc = img_raw.copy()

    # ── Segmentation — [FIX] array direct, pas de roundtrip disque ───
    seg_res = run_segmentation_array(
        img_uint8    = img_proc,
        model        = seg_model,
        device       = seg_device,
        threshold    = SEG_THR,
        tta          = SEG_TTA,
        cluster_dist = CLUSTER_DIST,
    )
    print(f'  Seg: {len(seg_res.raw_bboxes)} raw MC  →  {len(seg_res.bboxes)} clusters  '
          f'({seg_res.inference_ms:.0f}ms)')

    # ── Classification + Grad-CAM per cluster ─────────────────────────
    regions   = []
    crops     = []
    cams      = []
    gradcam_engine = GradCAM(clf.model, clf.model.net.features[7])

    for bb in seg_res.bboxes:
        crop = _crop_bbox(img_proc, bb, pad=32)
        crops.append(crop)

        # TTA-16 classification
        prob, label_int, per_tta = clf.predict_array(crop, return_all_probs=True)
        label_str   = 'MALIGNANT' if label_int == 1 else 'BENIGN'
        uncertainty = float(np.std(per_tta))

        # Grad-CAM (single pass, no TTA)
        img_f  = crop.astype(np.float32) / 255.0
        img_3c = np.stack([img_f]*3, axis=0)
        img_3c = (img_3c - _MEAN[:,None,None]) / _STD[:,None,None]
        tensor = torch.from_numpy(img_3c).unsqueeze(0).float().to(
            torch.device(device_str)
        )
        cam         = gradcam_engine.generate(tensor, out_size=(224, 224))
        cam_overlay = _make_gradcam_overlay(crop, cam, alpha=0.5)
        cams.append((cam, cam_overlay))

        print(f'    R{bb.region_id:02d} [{bb.x1},{bb.y1}→{bb.x2},{bb.y2}] '
              f'→ {label_str} (P={prob:.3f} ±{uncertainty:.3f})')

        regions.append(RegionResult(
            region_id   = bb.region_id,
            bbox        = bb.as_list,
            label       = label_str,
            prob        = prob,
            uncertainty = uncertainty,
            cam_overlay = cam_overlay,
        ))

    gradcam_engine.remove()

    elapsed = time.perf_counter() - t0
    print(f'  Total: {elapsed:.1f}s')

    all_results.append(dict(
        name     = img_path.name,
        stem     = img_path.stem,
        img_raw  = img_raw,
        img_proc = img_proc,
        seg_res  = seg_res,
        regions  = regions,
        crops    = crops,
        cams     = cams,
        elapsed  = elapsed,
    ))

print(f'\n✅ Done — {len(all_results)} image(s) processed')

## 3 · Visualisation par image
Pour chaque image : preprocessing → segmentation → classification → Grad-CAM → résultat final.

In [ ]:
def show_pipeline_steps(res: dict, save_path=None):
    """Affiche les 6 étapes du pipeline pour une image."""
    img_raw  = res['img_raw']
    img_proc = res['img_proc']
    seg_res  = res['seg_res']
    regions  = res['regions']
    name     = res['name']

    fig = plt.figure(figsize=(20, 10), facecolor='white')
    fig.suptitle(f'Pipeline complet — {name}',
                 color='black', fontsize=14, y=1.00)

    gs = gridspec.GridSpec(2, 4, figure=fig,
                           hspace=0.35, wspace=0.05,
                           left=0.02, right=0.98,
                           top=0.93, bottom=0.02)

    TITLE_COLOR = 'black'

    # ── [0,0] Image brute ──
    ax0 = fig.add_subplot(gs[0, 0])
    ax0.imshow(img_raw, cmap='gray', vmin=0, vmax=255)
    ax0.set_title('① Image brute', color=TITLE_COLOR, fontsize=10)
    ax0.axis('off')

    # ── [0,1] Prétraitée ──
    ax1 = fig.add_subplot(gs[0, 1])
    ax1.imshow(img_proc, cmap='gray', vmin=0, vmax=255)
    ax1.set_title('② Prétraitée (flip+CLAHE)', color=TITLE_COLOR, fontsize=10)
    ax1.axis('off')

    # ── [0,2] Probability map ──
    ax2 = fig.add_subplot(gs[0, 2])
    im2 = ax2.imshow(seg_res.prob_map, cmap='hot', vmin=0, vmax=1)
    ax2.set_title(
        f'③ U-Net prob map\n'
        f'{len(seg_res.raw_bboxes)} → {len(seg_res.bboxes)} clusters',
        color=TITLE_COLOR, fontsize=10
    )
    ax2.axis('off')

    cbar = plt.colorbar(im2, ax=ax2, fraction=0.035, pad=0.01)
    cbar.ax.tick_params(color='black', labelcolor='black')

    for bb in seg_res.bboxes:
        rect = mpatches.Rectangle(
            (bb.x1, bb.y1), bb.width, bb.height,
            linewidth=1.5, edgecolor='green', facecolor='none'
        )
        ax2.add_patch(rect)

    # ── [0,3] Masque ──
    ax3 = fig.add_subplot(gs[0, 3])
    ax3.imshow(seg_res.binary_mask, cmap='gray', vmin=0, vmax=255)
    ax3.set_title(f'④ Masque binaire (thr={SEG_THR})',
                  color=TITLE_COLOR, fontsize=10)
    ax3.axis('off')

    # ── [1,0] Overlay ──
    ax4 = fig.add_subplot(gs[1, 0])
    prob_u8  = (seg_res.prob_map * 255).astype(np.uint8)
    heatmap  = cv2.applyColorMap(prob_u8, cv2.COLORMAP_JET)
    canvas   = cv2.cvtColor(img_proc, cv2.COLOR_GRAY2BGR)
    overlay  = cv2.addWeighted(canvas, 0.6, heatmap, 0.4, 0)

    for bb in seg_res.bboxes:
        cv2.rectangle(overlay, (bb.x1, bb.y1), (bb.x2, bb.y2), (0,255,0), 2)
        cv2.putText(overlay, f'C{bb.region_id}',
                    (bb.x1+2, bb.y1+14),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0,255,0), 1)

    ax4.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    ax4.set_title('⑤ Segmentation overlay + clusters',
                  color=TITLE_COLOR, fontsize=10)
    ax4.axis('off')

    # ── [1,1] Classification ──
    ax5 = fig.add_subplot(gs[1, 1])
    ax5.set_facecolor('white')

    if regions:
        ids    = [f'R{r.region_id}' for r in regions]
        probs  = [r.prob for r in regions]
        uncert = [r.uncertainty for r in regions]

        colors = ['red' if r.label=='MALIGNANT' else 'gold'
                  for r in regions]

        y_pos  = range(len(regions))
        bars   = ax5.barh(list(y_pos), probs, color=colors, height=0.5,
                          xerr=uncert, error_kw=dict(color='black', capsize=4))

        ax5.set_yticks(list(y_pos))
        ax5.set_yticklabels(ids, color='black', fontsize=9)
        ax5.set_xlim(0, 1)

        ax5.axvline(clf.threshold, color='black', ls='--', lw=1.2)

        ax5.set_xlabel('P(malignant)', color='black')
        ax5.set_title('⑥ Classification TTA-16',
                      color=TITLE_COLOR, fontsize=10)

        ax5.tick_params(colors='black')

        for spine in ax5.spines.values():
            spine.set_color('#999')

        for bar, r in zip(bars, regions):
            ax5.text(min(r.prob + r.uncertainty + 0.02, 0.97),
                     bar.get_y() + bar.get_height()/2,
                     f'{r.prob:.3f}', va='center', color='black', fontsize=8)
    else:
        ax5.text(0.5, 0.5, 'Aucun cluster détecté',
                 ha='center', va='center', color='gray',
                 transform=ax5.transAxes, fontsize=11)
        ax5.axis('off')

    # ── [1,2] Résultat final ──
    ax6 = fig.add_subplot(gs[1, 2])
    final = cv2.cvtColor(img_proc, cv2.COLOR_GRAY2BGR)
    final = _annotate_image(final, regions, show_prob=True, show_gradcam=False)
    final = _build_legend(final)

    n_mal = sum(1 for r in regions if r.label=='MALIGNANT')
    n_ben = sum(1 for r in regions if r.label=='BENIGN')

    cv2.putText(final, f'MAL={n_mal}  BEN={n_ben}',
                (8, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 2)

    ax6.imshow(cv2.cvtColor(final, cv2.COLOR_BGR2RGB))
    ax6.set_title('⑦ Résultat final (RED=Malin, YELLOW=Bénin)',
                  color=TITLE_COLOR, fontsize=10)
    ax6.axis('off')

    # ── [1,3] Stats ──
    ax7 = fig.add_subplot(gs[1, 3])
    ax7.set_facecolor('white')
    ax7.axis('off')

    lines = [
        ('Image', name, 'black'),
        ('Clusters', str(len(seg_res.bboxes)), 'black'),
        ('Malignants', str(n_mal), 'red' if n_mal>0 else 'black'),
        ('Bénins', str(n_ben), 'gold' if n_ben>0 else 'black'),
        ('Temps total', f'{res["elapsed"]:.1f}s', 'black'),
    ]

    y = 0.9
    for label, value, color in lines:
        ax7.text(0.05, y, f'{label}: {value}', color=color, fontsize=10)
        y -= 0.1

    # ── Save ──
    if save_path:
        plt.savefig(str(save_path), dpi=140, bbox_inches='tight',
                    facecolor='white')
        print(f'  Saved: {save_path}')

    plt.show()


# Appel
for res in all_results:
    show_pipeline_steps(
        res,
        save_path=OUTPUT_DIR / f"{res['stem']}_pipeline_steps.png"
    )

## 4 · Détail par cluster — Grad-CAM + distribution TTA

In [ ]:
def show_cluster_detail(res: dict, save_path=None):
    """Pour chaque cluster : crop brut, Grad-CAM, distribution TTA des 16 vues."""
    regions = res['regions']
    crops   = res['crops']
    cams    = res['cams']
    name    = res['name']

    if not regions:
        print(f'  {name}: aucun cluster détecté — skip')
        return

    n  = len(regions)
    fig, axes = plt.subplots(n, 4,
                              figsize=(16, 4 * n),
                              facecolor='#0d1117',
                              squeeze=False)
    fig.suptitle(f'Détail par cluster — {name}',
                 color='white', fontsize=13, y=1.00)

    _MEAN_np = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    _STD_np  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    for i, (r, crop, (cam, cam_ov)) in enumerate(
            zip(regions, crops, cams)):

        color_str = '#FF4444' if r.label == 'MALIGNANT' else '#FFD700'

        # ── Col 0: Crop brut ──────────────────────────────────────────
        ax = axes[i][0]
        ax.imshow(crop, cmap='gray', vmin=0, vmax=255)
        ax.set_title(
            f'R{r.region_id} — {r.label}\n'
            f'bbox={r.bbox}',
            color=color_str, fontsize=9
        )
        ax.axis('off')
        ax.set_facecolor('#0d1117')
        for sp in ax.spines.values():
            sp.set_visible(True); sp.set_color(color_str); sp.set_linewidth(3)

        # ── Col 1: Grad-CAM overlay ────────────────────────────────────
        ax = axes[i][1]
        ax.imshow(cv2.cvtColor(cam_ov, cv2.COLOR_BGR2RGB))
        ax.set_title('Grad-CAM (net.features[7])',
                     color='#aaaaaa', fontsize=9)
        ax.axis('off')

        # ── Col 2: Grad-CAM heatmap seul ─────────────────────────────
        ax = axes[i][2]
        im = ax.imshow(cam, cmap='jet', vmin=0, vmax=1)
        ax.set_title('Grad-CAM (raw, normalisé)',
                     color='#aaaaaa', fontsize=9)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.04, pad=0.01)

        # ── Col 3: Distribution des 16 probabilités TTA ───────────────
        ax = axes[i][3]
        ax.set_facecolor('#161b22')

        # Re-run TTA pour récupérer les 16 probs individuelles
        _, _, per_tta = clf.predict_array(crop, return_all_probs=True)
        tta_labels = [f'T{j+1:02d}' for j in range(len(per_tta))]
        colors_tta = ['#FF4444' if p >= clf.threshold else '#4488FF'
                      for p in per_tta]

        ax.bar(range(len(per_tta)), per_tta, color=colors_tta, width=0.7)
        ax.axhline(clf.threshold, color='white', ls='--', lw=1.2,
                   label=f'thr={clf.threshold:.2f}')
        ax.axhline(r.prob, color=color_str, ls='-', lw=1.5,
                   label=f'mean={r.prob:.3f}')
        ax.fill_between(
            range(len(per_tta)),
            r.prob - r.uncertainty,
            r.prob + r.uncertainty,
            color=color_str, alpha=0.15
        )
        ax.set_ylim(0, 1)
        ax.set_xticks(range(0, len(per_tta), 2))
        ax.set_xticklabels([f'T{j+1}' for j in range(0, len(per_tta), 2)],
                           color='white', fontsize=7)
        ax.tick_params(axis='y', colors='white')
        ax.set_xlabel('Vue TTA', color='#888')
        ax.set_ylabel('P(malignant)', color='#888')
        ax.set_title(
            f'Distribution TTA-{len(per_tta)}\n'
            f'mean={r.prob:.3f}  std={r.uncertainty:.3f}',
            color='#aaaaaa', fontsize=9
        )
        ax.legend(fontsize=7)
        for sp in ax.spines.values(): sp.set_color('#444')

    plt.tight_layout()
    if save_path:
        plt.savefig(str(save_path), dpi=130, bbox_inches='tight',
                    facecolor='#0d1117')
        print(f'  Saved: {save_path}')
    plt.show()


for res in all_results:
    show_cluster_detail(
        res,
        save_path=OUTPUT_DIR / f"{res['stem']}_cluster_detail.png"
    )

## 5 · Vue comparative multi-images

In [ ]:
if len(all_results) > 1:
    n   = len(all_results)
    fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n),
                              facecolor='#0d1117', squeeze=False)
    fig.suptitle('Comparaison multi-images — mammo-cad',
                 color='white', fontsize=14, y=1.00)

    for row, res in enumerate(all_results):
        img_proc = res['img_proc']
        seg_res  = res['seg_res']
        regions  = res['regions']
        n_mal    = sum(1 for r in regions if r.label=='MALIGNANT')
        n_ben    = sum(1 for r in regions if r.label=='BENIGN')
        label_c  = '#FF4444' if n_mal>0 else '#FFD700' if n_ben>0 else '#00cc66'
        assess   = 'SUSPICIOUS' if n_mal>0 else 'BENIGN' if n_ben>0 else 'NORMAL'

        # Col 0: preprocessed
        ax = axes[row][0]
        ax.imshow(img_proc, cmap='gray')
        ax.set_title(f'{res["name"]}\nPreprocessed',
                     color='white', fontsize=9)
        ax.axis('off')

        # Col 1: prob map + clusters
        ax = axes[row][1]
        ax.imshow(seg_res.prob_map, cmap='hot', vmin=0, vmax=1)
        for bb in seg_res.bboxes:
            rect = mpatches.Rectangle(
                (bb.x1,bb.y1), bb.width, bb.height,
                lw=1.5, edgecolor='#00ff88', facecolor='none'
            )
            ax.add_patch(rect)
        ax.set_title(
            f'U-Net: {len(seg_res.bboxes)} clusters',
            color='white', fontsize=9
        )
        ax.axis('off')

        # Col 2: final annotation
        ax = axes[row][2]
        final = cv2.cvtColor(img_proc, cv2.COLOR_GRAY2BGR)
        final = _annotate_image(final, regions, show_prob=True, show_gradcam=False)
        ax.imshow(cv2.cvtColor(final, cv2.COLOR_BGR2RGB))
        ax.set_title(
            f'Résultat: {assess}\nMAL={n_mal}  BEN={n_ben}  ({res["elapsed"]:.1f}s)',
            color=label_c, fontsize=9
        )
        ax.axis('off')
        for sp in ax.spines.values():
            sp.set_visible(True); sp.set_color(label_c); sp.set_linewidth(3)

    plt.tight_layout()
    out = OUTPUT_DIR / 'multi_image_comparison.png'
    plt.savefig(str(out), dpi=130, bbox_inches='tight', facecolor='#0d1117')
    print(f'Saved: {out}')
    plt.show()
else:
    print('Une seule image — section multi-images ignorée.')

## 6 · Analyse de la carte de segmentation — profil de probabilité

In [ ]:
def show_seg_analysis(res: dict, save_path=None):
    """Analyse détaillée de la carte de probabilité U-Net."""
    prob_map = res['seg_res'].prob_map
    binary   = res['seg_res'].binary_mask
    bboxes   = res['seg_res'].bboxes
    name     = res['name']

    fig, axes = plt.subplots(2, 3, figsize=(15, 9), facecolor='#0d1117')
    fig.suptitle(f'Analyse segmentation — {name}',
                 color='white', fontsize=13, y=1.00)

    # ── [0,0] Prob map ────────────────────────────────────────────────
    ax = axes[0][0]
    im = ax.imshow(prob_map, cmap='hot', vmin=0, vmax=1)
    ax.set_title('Carte de probabilité U-Net', color='#aaa', fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.04)

    # ── [0,1] Distribution des probabilités ───────────────────────────
    ax = axes[0][1]
    ax.set_facecolor('#161b22')
    probs_flat = prob_map.flatten()
    # Zoom sur les pixels > 0.1 (ignorer le fond)
    probs_pos  = probs_flat[probs_flat > 0.05]
    ax.hist(probs_pos, bins=80, color='#FF6B35', alpha=0.8, density=True)
    ax.axvline(SEG_THR, color='white', ls='--', lw=1.5,
               label=f'seuil={SEG_THR}')
    above_thr = (probs_flat >= SEG_THR).sum()
    ax.set_xlabel('P(MC)', color='white')
    ax.set_ylabel('Densité', color='white')
    ax.set_title(
        f'Distribution P(MC) [pixels > 0.05]\n'
        f'{above_thr} pixels ≥ seuil ({100*above_thr/len(probs_flat):.2f}%)',
        color='#aaa', fontsize=9
    )
    ax.legend(fontsize=8)
    ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_color('#444')

    # ── [0,2] Masque binaire ──────────────────────────────────────────
    ax = axes[0][2]
    ax.imshow(binary, cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'Masque binaire (thr={SEG_THR})\n'
                 f'{len(bboxes)} clusters après filtrage',
                 color='#aaa', fontsize=9)
    ax.axis('off')

    # ── [1,0] Profil horizontal (ligne médiane) ────────────────────────
    ax = axes[1][0]
    ax.set_facecolor('#161b22')
    H, W  = prob_map.shape
    mid_h = H // 2
    ax.plot(prob_map[mid_h, :], color='#FF6B35', lw=1.2)
    ax.axhline(SEG_THR, color='white', ls='--', lw=1, label=f'seuil={SEG_THR}')
    ax.fill_between(range(W), 0, prob_map[mid_h, :],
                    where=(prob_map[mid_h,:] >= SEG_THR),
                    color='#FF4444', alpha=0.4, label='MC détecté')
    ax.set_xlim(0, W)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Colonne pixel', color='white')
    ax.set_ylabel('P(MC)', color='white')
    ax.set_title('Profil horizontal (ligne médiane)', color='#aaa', fontsize=9)
    ax.legend(fontsize=7)
    ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_color('#444')

    # ── [1,1] Profil vertical (colonne médiane) ────────────────────────
    ax = axes[1][1]
    ax.set_facecolor('#161b22')
    mid_w = W // 2
    ax.plot(prob_map[:, mid_w], range(H), color='#4DA1FF', lw=1.2)
    ax.axvline(SEG_THR, color='white', ls='--', lw=1, label=f'seuil={SEG_THR}')
    ax.fill_betweenx(range(H), 0, prob_map[:, mid_w],
                     where=(prob_map[:,mid_w] >= SEG_THR),
                     color='#FF4444', alpha=0.4, label='MC détecté')
    ax.set_ylim(H, 0)   # invert y for image convention
    ax.set_xlim(0, 1)
    ax.set_xlabel('P(MC)', color='white')
    ax.set_ylabel('Ligne pixel', color='white')
    ax.set_title('Profil vertical (colonne médiane)', color='#aaa', fontsize=9)
    ax.legend(fontsize=7)
    ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_color('#444')

    # ── [1,2] Carte de confiance (prob > 0.7) ────────────────────────
    ax = axes[1][2]
    high_conf = (prob_map >= 0.7).astype(np.uint8) * 255
    ax.imshow(high_conf, cmap='hot', vmin=0, vmax=255)
    ax.set_title('Zones haute confiance (P ≥ 0.70)',
                 color='#aaa', fontsize=9)
    ax.axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(str(save_path), dpi=130, bbox_inches='tight',
                    facecolor='#0d1117')
        print(f'  Saved: {save_path}')
    plt.show()


for res in all_results:
    show_seg_analysis(
        res,
        save_path=OUTPUT_DIR / f"{res['stem']}_seg_analysis.png"
    )

## 7 · Rapport JSON complet

In [ ]:
all_reports = []

for res in all_results:
    regions = res['regions']
    seg_res = res['seg_res']
    n_mal   = sum(1 for r in regions if r.label=='MALIGNANT')
    n_ben   = sum(1 for r in regions if r.label=='BENIGN')

    report = {
        'image'             : res['name'],
        'model_auc_cls'     : 0.7812,
        'model_auc_seg'     : 0.9642,
        'regions_found'     : len(regions),
        'malignant'         : n_mal,
        'benign'            : n_ben,
        'raw_mc_detections' : len(seg_res.raw_bboxes),
        'seg_threshold'     : SEG_THR,
        'cls_threshold'     : clf.threshold,
        'tta_cls'           : CLS_TTA,
        'tta_seg'           : SEG_TTA,
        'inference_s'       : round(res['elapsed'], 2),
        'overall_assessment': (
            'SUSPICIOUS — malignant cluster(s) detected' if n_mal > 0
            else 'BENIGN — no malignant clusters' if n_ben > 0
            else 'NORMAL — no microcalcification clusters detected'
        ),
        'results': [
            {
                'region_id'  : r.region_id,
                'location'   : r.bbox,
                'label'      : r.label,
                'probability': round(r.prob, 4),
                'uncertainty': round(r.uncertainty, 4),
                'confidence' : f'{r.prob*100:.1f}%',
            }
            for r in regions
        ],
    }
    all_reports.append(report)

    # Save individual report
    rpath = OUTPUT_DIR / f"{res['stem']}_report.json"
    rpath.write_text(json.dumps(report, indent=2))
    print(f'\n── {res["name"]} ──')
    print(json.dumps(report, indent=2))

# Save summary report
summary_path = OUTPUT_DIR / 'pipeline_summary.json'
summary_path.write_text(json.dumps(all_reports, indent=2))
print(f'\n✅ Summary report: {summary_path}')

## 8 · Récapitulatif des outputs générés

In [ ]:
print(f'Output directory: {OUTPUT_DIR}\n')
all_files = sorted(OUTPUT_DIR.rglob('*'))
total_mb  = sum(f.stat().st_size for f in all_files if f.is_file()) / 1e6

ext_icons = {
    '.png':  '🖼',
    '.json': '📋',
    '.txt':  '📄',
}
for f in all_files:
    if f.is_file():
        icon = ext_icons.get(f.suffix, '📁')
        size = f.stat().st_size / 1e3
        print(f'  {icon} {f.relative_to(OUTPUT_DIR)}  ({size:.0f} KB)')

print(f'\nTotal: {len([f for f in all_files if f.is_file()])} fichiers  —  {total_mb:.1f} MB')